<a href="https://colab.research.google.com/github/TalCordova/RMBA_SemB26_TC_SC/blob/main/Final_Project_SC_TC_fit_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train Notebook

In this notebook we will use the conclusions from the [EDA Notebooks](https://colab.research.google.com/drive/1tDrZcNQwQODdfNNtE4JyiLV2xT8t-kex#scrollTo=vD4I5v_ltkDW).

In [7]:
import pandas as pd
from sklearn.metrics import make_scorer

In [1]:
# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
tal_path_ffp_train = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/ffp_train.csv'
tal_path_reviews_train = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/reviews_training.csv'

saar_path_ffp_train ='/content/drive/MyDrive/Research Methods for Business Analytics/ffp_train.csv'
saar_pth_ffp_reviews = '/content/drive/MyDrive/Research Methods for Business Analytics/reviews_training.csv'
# --- Load training data for EDA ---
ffp_train = pd.read_csv(tal_path_ffp_train)
reviews_train = pd.read_csv(tal_path_reviews_train)

# --- Initial shape check ---
print(f'ffp_train shape: {ffp_train.shape}')
print(f'reviews_train shape: {reviews_train.shape}')

ffp_train shape: (20000, 23)
reviews_train shape: (1499, 2001)


## Feature Engineering

We will process the dataset as follows:

1. Consolidate all the `FARE` and `POINTS` features - these features are highly correlated, and there is no need for them all. We want to still be able to reflect the numeric magnitude, and also reflect the importance of recency. Due to the nature of the features, we decided to create a weighted sum based on recency - the most recent purchases will get a weight 1, then 0.8 all the way to 0.2

2. Remove the `ANIMAL_FLAG` - the EDA pointed it out to be practically useless discriminator.

In [5]:
# --- Feature Engineering: Recency-Weighted Fare & Points ---
RECENCY_WEIGHTS = [1.0, 0.8, 0.6, 0.4, 0.2]  # Y1 (most recent) → Y5 (oldest)

fare_cols   = [f'FARE_L_Y{i}'   for i in range(1, 6)]
points_cols = [f'POINTS_L_Y{i}' for i in range(1, 6)]

ffp_train['FARE_WEIGHTED']   = sum(
    w * ffp_train[c] for w, c in zip(RECENCY_WEIGHTS, fare_cols)
)
ffp_train['POINTS_WEIGHTED'] = sum(
    w * ffp_train[c] for w, c in zip(RECENCY_WEIGHTS, points_cols)
)

# --- Drop original yearly columns and ANIMAL_FLAG ---
cols_to_drop = fare_cols + points_cols + ['ANIMAL_FLAG']
ffp_train.drop(columns=cols_to_drop, inplace=True)

# --- Also drop EDA-only aggregates created earlier (FARE_MEAN, POINTS_MEAN) ---
ffp_train.drop(columns=['FARE_MEAN', 'POINTS_MEAN'], inplace=True, errors='ignore')

# --- Verify final feature set ---
print('Remaining columns:', ffp_train.columns.tolist())
print('Shape:', ffp_train.shape)

Remaining columns: ['ID', 'CUSTOMER_GRADE', 'STATUS_PANTINUM', 'STATUS_GOLD', 'STATUS_SILVER', 'NUM_DEAL', 'LAST_DEAL', 'ADVANCE_PURCHASE', 'ATT_FLAG', 'CREDIT_FLAG', 'RELATED_FLAG', 'BUYER_FLAG', 'FARE_WEIGHTED', 'POINTS_WEIGHTED']
Shape: (20000, 14)


## Custom Metric

As we mentioned in the previous notebok, and in the project report the dataset is highly imbalanced.

In addition, we have assymetric error, since a TP is worth $78.4 and on a Fp we lose $15.2 - therefore, we will use the cutom metric:

$Profit = 78.4TP - 15.2FP$

In [8]:
THRESHOLD = 0.1624

feature_cols = [c for c in ffp_train.columns if c not in ['ID', 'BUYER_FLAG']]

X = ffp_train[feature_cols]
y = ffp_train['BUYER_FLAG']

# --- Custom profit scorer ---
def avg_profit(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (y_prob >= threshold).astype(int)
    TP = ((y_pred == 1) & (y_true == 1)).sum()
    FP = ((y_pred == 1) & (y_true == 0)).sum()
    return (TP * 78.4 - FP * 15.2) / len(y_true)

profit_scorer = make_scorer(avg_profit, needs_proba=True, response_method='predict_proba')

## Fit Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.metrics import make_scorer, roc_auc_score, accuracy_score
import numpy as np

# --- Constants ---
THRESHOLD = 0.1624
RANDOM_STATE = 42

feature_cols = [c for c in ffp_train.columns if c not in ['ID', 'BUYER_FLAG']]
X = ffp_train[feature_cols]
y = ffp_train['BUYER_FLAG']

# --- Custom profit scorer ---
def avg_profit_scorer(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (y_prob >= threshold).astype(int)
    TP = ((y_pred == 1) & (y_true == 1)).sum()
    FP = ((y_pred == 1) & (y_true == 0)).sum()
    return (TP * 78.4 - FP * 15.2) / len(y_true)

profit_scorer = make_scorer(avg_profit_scorer, needs_proba=True, response_method='predict_proba')
auc_scorer    = make_scorer(roc_auc_score, needs_proba=True, response_method='predict_proba')

scoring = {
    'avg_profit': profit_scorer,
    'auc':        auc_scorer,
    'accuracy':   'accuracy',
}

# --- CV strategy: stratified to preserve 9.5% minority class in each fold ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# --- Model definitions ---
# LR needs scaling; trees do not — each wrapped in its own Pipeline
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Decision Tree': Pipeline([
        ('clf', DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    'Random Forest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))
        # n_jobs=-1 omitted — causes deadlocks in Colab
    ]),
    'XGBoost': Pipeline([
        ('clf', XGBClassifier(
            n_estimators=100,
            eval_metric='logloss',
            random_state=RANDOM_STATE,
            verbosity=0
        ))
    ]),
}

# --- Run CV for all models ---
results = {}

for name, pipeline in models.items():
    print(f'Running CV: {name}...')
    cv_results = cross_validate(
        pipeline, X, y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )
    results[name] = {
        'avg_profit_mean': cv_results['test_avg_profit'].mean(),
        'avg_profit_std':  cv_results['test_avg_profit'].std(),
        'auc_mean':        cv_results['test_auc'].mean(),
        'auc_std':         cv_results['test_auc'].std(),
        'accuracy_mean':   cv_results['test_accuracy'].mean(),
        'accuracy_std':    cv_results['test_accuracy'].std(),
    }
    print(f"  Avg Profit: {results[name]['avg_profit_mean']:.4f} ± {results[name]['avg_profit_std']:.4f}")
    print(f"  AUC:        {results[name]['auc_mean']:.4f} ± {results[name]['auc_std']:.4f}")
    print(f"  Accuracy:   {results[name]['accuracy_mean']:.4f} ± {results[name]['accuracy_std']:.4f}")
    print()

# --- Summary table ---
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('avg_profit_mean', ascending=False)
print('=== Baseline Model Comparison (sorted by Avg Profit) ===')
print(results_df.round(4))